# Dueling DQN — Dueling Network Architectures for Deep RL (2016)

_Papers · Reinforcement Learning_

**Split the Q-network's head into a state-value stream V(s) and an advantage stream A(s,a), then recombine them — so the network learns which states are valuable without having to learn the effect of every action there.**

---

This notebook builds a dueling DQN from scratch, one piece at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q gymnasium
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — gymnasium + PyTorch (Colab)

We build a dueling DQN and train it on CartPole, alongside a plain single-stream DQN for an honest ablation. We go in four steps: (1) the aggregation arithmetic of Eq. 9, (2) the Q-network with a switchable head, (3) the DQN training loop, (4) running both and comparing how fast each solves the task.

### Step 1 — Check the Eq. 9 aggregation by hand

The dueling head splits the output into a state-value `V(s)` and per-action advantages `A(s, a)`, then recombines them as `Q = V + (A - mean(A))`. Subtracting the per-state mean advantage is what makes the split unique (identifiable). Here we confirm the lesson's worked example: with `V = 12` and `A = [2, -4]`, the centered advantage is `[3, -3]`, so `Q = [15, 9]` and `mean(Q) = 12 = V`.

In [ ]:
# In Colab first run:  !pip install gymnasium
# (torch is preinstalled in Colab; gymnasium is not.)
import random, collections
import torch
import torch.nn as nn
import gymnasium as gym

torch.manual_seed(0)
random.seed(0)

# --- 0. Sanity-check the lesson's worked example of Eq. 9. ---
V = torch.tensor([[12.0]])              # value stream V(s), shape [1, 1]
A = torch.tensor([[2.0, -4.0]])         # advantage stream A(s, .), shape [1, 2]
A_centered = A - A.mean(dim=1, keepdim=True)   # subtract per-state mean over actions
Q = V + A_centered                      # Eq. 9
print("worked example: mean(A) =", A.mean().item(),
      " centered =", A_centered.tolist(),
      " Q =", Q.tolist(), " mean(Q) =", Q.mean().item())
# worked example: mean(A) = -1.0  centered = [[3.0, -3.0]]  Q = [[15.0, 9.0]]  mean(Q) = 12.0

### Step 2 — Build a Q-network with a switchable head

The body (two ReLU layers) is shared. The head is switchable: when `dueling=True` it has a **value head** (one scalar) and an **advantage head** (one per action), recombined with Eq. 9 in the forward pass. When `dueling=False` it's a single linear layer that outputs `Q(s, a)` directly — the ablation baseline. Only the head differs between the two.

In [ ]:
# --- 1. Q-network with a SWITCHABLE head: dueling (Eq. 9) vs plain single-stream. ---
class QNet(nn.Module):
    def __init__(self, obs_dim, n_act, hidden=128, dueling=True):
        super().__init__()
        self.dueling = dueling
        self.body = nn.Sequential(nn.Linear(obs_dim, hidden), nn.ReLU(),
                                  nn.Linear(hidden, hidden), nn.ReLU())
        if dueling:
            self.value_head = nn.Linear(hidden, 1)        # V(s)   -> scalar per state
            self.adv_head   = nn.Linear(hidden, n_act)    # A(s,a) -> one per action
        else:
            self.q_head     = nn.Linear(hidden, n_act)    # ABLATION: Q(s,a) directly

    def forward(self, x):
        h = self.body(x)
        if self.dueling:
            v = self.value_head(h)                         # [batch, 1]
            a = self.adv_head(h)                           # [batch, n_act]
            return v + (a - a.mean(dim=1, keepdim=True))   # Eq. 9: center then add V
        return self.q_head(h)                              # plain single-stream Q

### Step 3 — Write the standard DQN training loop

This is ordinary DQN scaffolding, kept **identical** between the dueling and plain variants: an epsilon-greedy policy, a replay buffer, an online and a target network, and a TD mean-squared-error loss with gradient-norm clipping. The only thing that ever changes is the head shape (the `dueling` flag passed to `QNet`), so any difference in learning speed is attributable to the head.

In [ ]:
# --- 2. Standard DQN scaffolding (UNCHANGED between dueling and plain). ---
def train(dueling=True, episodes=300, gamma=0.99, batch=64,
          buf_cap=10000, lr=1e-3, target_sync=200):
    torch.manual_seed(0); random.seed(0)
    env = gym.make("CartPole-v1")
    obs_dim = env.observation_space.shape[0]
    n_act = env.action_space.n
    online = QNet(obs_dim, n_act, dueling=dueling)
    target = QNet(obs_dim, n_act, dueling=dueling)
    target.load_state_dict(online.state_dict())
    opt = torch.optim.Adam(online.parameters(), lr=lr)
    buf = collections.deque(maxlen=buf_cap)
    eps, eps_min, eps_decay = 1.0, 0.05, 0.995
    step = 0
    history = []

    for ep in range(episodes):
        obs, _ = env.reset(seed=ep)
        done = False
        ep_ret = 0.0
        while not done:
            step += 1
            if random.random() < eps:                      # epsilon-greedy exploration
                a = env.action_space.sample()
            else:
                with torch.no_grad():
                    a = int(online(torch.as_tensor(obs, dtype=torch.float32)
                                   .unsqueeze(0)).argmax(1))
            nobs, rew, term, trunc, _ = env.step(a)
            done = term or trunc
            buf.append((obs, a, rew, nobs, float(done)))
            obs = nobs
            ep_ret += rew

            if len(buf) >= batch:                          # one gradient step
                S, Acs, Rs, NS, Ds = zip(*random.sample(buf, batch))
                S  = torch.as_tensor(S,  dtype=torch.float32)
                NS = torch.as_tensor(NS, dtype=torch.float32)
                Acs = torch.as_tensor(Acs).long()
                Rs  = torch.as_tensor(Rs, dtype=torch.float32)
                Ds  = torch.as_tensor(Ds, dtype=torch.float32)
                q = online(S).gather(1, Acs.unsqueeze(1)).squeeze(1)   # Q(s, a)
                with torch.no_grad():
                    q_next = target(NS).max(1).values             # max_a' Q_target(s', a')
                    tgt = Rs + gamma * q_next * (1 - Ds)           # TD target
                loss = nn.functional.mse_loss(q, tgt)              # TD squared error
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(online.parameters(), 10)  # paper: grad norm <= 10
                opt.step()
                if step % target_sync == 0:
                    target.load_state_dict(online.state_dict())
        eps = max(eps_min, eps * eps_decay)
        history.append(ep_ret)
        if ep % 20 == 0:
            recent = sum(history[-20:]) / len(history[-20:])
            print(f"  ep {ep:3d}  return {ep_ret:5.0f}  avg20 {recent:6.1f}  eps {eps:.2f}")
    env.close()
    return history

### Step 4 — Train both heads and compare time-to-solve

Run the loop twice — dueling head, then plain single-stream head — with everything else fixed. `first_solved` reports the first episode where the trailing-20-episode average return crosses the 195 solved threshold. The dueling head typically gets there in fewer episodes, because sharing `V(s)` across actions lets one update improve every action's value at once.

In [ ]:
print("DUELING DQN (Eq. 9 split head):")
duel_hist = train(dueling=True)
print("\nABLATION -- plain single-stream Q head (same everything else):")
plain_hist = train(dueling=False)

def first_solved(h, thresh=195, window=20):
    for i in range(window, len(h)):
        if sum(h[i-window:i]) / window >= thresh:
            return i
    return None

print("\nFirst episode with 20-ep avg >= 195:")
print("  dueling:", first_solved(duel_hist), " plain:", first_solved(plain_hist))
# Dueling typically crosses the solved threshold in fewer episodes than the plain head.
# (Exact numbers vary by hardware/seed; our small run, not the paper's Atari numbers.)

## Visualize the data & results

_Does splitting the Q-head into a value stream V(s) and an advantage stream A(s,a), recombined by Eq. 9, make a DQN solve CartPole in fewer episodes than a plain single-stream head — with the replay buffer, target network, loss, and seed held identical?_

In [ ]:
# Sketch of how the two curves above are produced (full loop in the CODE cell).
# Train a dueling-head DQN and a plain single-stream DQN on CartPole-v1 with IDENTICAL
# body / replay buffer / target net / TD loss / lr / epsilon schedule / seed, recording
# the trailing-20-episode average return per episode.
#
#   duel_hist  = train(dueling=True)    # green: split head -> crosses solved line sooner
#   plain_hist = train(dueling=False)   # red:   single Q head -> same shape, lags behind
#
# The ONLY differing line is the head:
#   dueling:  q = v + (a - a.mean(dim=1, keepdim=True))   # Eq. 9
#   plain:    q = q_head(h)                                # single linear layer
#
# Sharing V(s) across actions generalizes faster, so dueling reaches the ~195 solved
# threshold in fewer episodes. (Numbers are from our own run; the paper reports Atari
# Table-1 human-normalized scores -- mean 373.1%, median 151.5% for Duel Clip -- not
# these CartPole values.)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The worked aggregation. A dueling Q-network on a 2-action state outputs value stream
            $V(s) = 12.0$ and raw advantage stream $A(s,\cdot) = [\,2.0,\,-4.0\,]$. Apply Eq. 9: compute the
            mean advantage, the centered advantages, the resulting $Q(s,\cdot)$, and verify that the mean of
            $Q$ equals $V$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Mean advantage: $\bar{A} = \tfrac{1}{2}(2.0 + (-4.0)) = -1.0$. — _Eq. 9 subtracts the per-state mean over the $|\mathcal{A}| = 2$ actions._
- Center: $A - \bar{A} = [\,2.0-(-1.0),\; -4.0-(-1.0)\,] = [\,3.0,\,-3.0\,]$. — _Re-centering makes the advantage vector sum to zero, fixing identifiability._
- Add the value: $Q = 12.0 + [\,3.0,\,-3.0\,] = [\,15.0,\,9.0\,]$. — _$V$ broadcasts across both actions (Eq. 9)._
- Check: $\tfrac{1}{2}(15.0 + 9.0) = 12.0 = V$. — _Centering forces $V$ to equal the action-mean of $Q$ — the unique split._

**Answer:** $\bar{A} = -1.0$, centered $[3.0, -3.0]$, $Q = [15.0, 9.0]$, and $\text{mean}(Q) = 12.0 = V$.
                 The greedy action is "left" ($15.0 \gt 9.0$), the same $\arg\max$ as the raw advantage. The
                 notebook recomputes V + (A - A.mean()) = $[15.0, 9.0]$.

</details>

**Problem 2.** The identifiability problem. The naive head (Eq. 7) outputs $V = 12.0$ and
            $A = [\,2.0,\,-4.0\,]$, giving $Q = [\,14.0,\,8.0\,]$. Show that a different $(V, A)$ pair gives the
            same $Q$, and explain why Eq. 9 removes this ambiguity.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Pick $c = 88$: set $V' = 12.0 + 88 = 100.0$ and $A' = [\,2.0-88,\,-4.0-88\,] = [\,-86.0,\,-92.0\,]$. — _Adding $c$ to $V$ and subtracting $c$ from every advantage is the shift that leaves the sum unchanged._
- Recompute: $V' + A' = 100.0 + [\,-86.0,\,-92.0\,] = [\,14.0,\,8.0\,]$ — identical $Q$. — _The constant cancels, so the network can't tell $(12, [2,-4])$ from $(100, [-86,-92])$._
- Now apply Eq. 9 to $A' = [-86, -92]$: $\bar{A'} = -89$, centered $= [3.0, -3.0]$, same as before. — _Centering subtracts the mean, which also dropped by $c$, so the centered advantage is invariant — the ambiguity is gone._

**Answer:** Both $(V=12, A=[2,-4])$ and $(V'=100, A'=[-86,-92])$ produce $Q = [14, 8]$ under the naive
                 Eq. 7, so $V$ and $A$ are not uniquely determined. Under Eq. 9 both collapse to the same centered
                 advantage $[3.0, -3.0]$ and force $V = \text{mean}(Q)$, so the split is unique. This is exactly
                 the identifiability fix the paper's &sect;3 describes.

</details>

**Problem 3.** The ablation. Your dueling DQN solves CartPole faster than a plain DQN. To prove the split is
            the cause, you collapse the dueling head to a single linear layer that outputs $Q$ directly, keeping
            the body, replay buffer, target network, loss, optimizer, and seed identical. What do you expect, and
            what does the result demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Replace the two heads + Eq. 9 with one nn.Linear(hidden, n_actions) outputting $Q$ directly; change nothing else. — _An honest ablation flips exactly one bit — the head shape — so any difference is attributable to the dueling split._
- Retrain and compare episode-return curves: the dueling agent reaches the solved threshold in fewer episodes; the plain head is slower / noisier. — _Sharing $V$ across actions lets one update improve every action's value at once, generalizing faster — especially where actions are similar-valued._
- Conclude the head shape, not the body or the RL algorithm, drives the speed-up. — _Both agents share every other component, isolating the dueling aggregation as the cause._

**Answer:** The dueling agent should climb to the solved score in fewer episodes than the single-stream
                 ablation, while both eventually reach a similar final policy (centering doesn't change the greedy
                 action). Since the only difference is Eq. 9's split head, this isolates the value/advantage
                 decomposition as the source of the sample-efficiency gain &mdash; matching the paper's "better
                 policy evaluation in the presence of many similar-valued actions." The CODEVIZ panel shows this
                 contrast on our small run.

</details>